## One Hot Encoding

# 🔢 One-Hot Encoding in NLP

**One-Hot Encoding** is a technique used to represent categorical data (words) as binary vectors. It is the most basic way to turn text into numbers.

---

## 💡 The Concept
In One-Hot Encoding, every unique word in your vocabulary becomes a **column**. 
- If the word exists in the document, the value is **1**.
- If the word does not exist, the value is **0**.

### Example:
**Corpus**:
1. "dog bites man"
2. "man bites dog"

**Vocabulary**: `[bites, dog, man]`

| Document | bites | dog | man |
| :--- | :---: | :---: | :---: |
| Doc 1 | 1 | 1 | 1 |
| Doc 2 | 1 | 1 | 1 |

*Note: In this specific example, the vectors for Doc 1 and Doc 2 are identical because One-Hot Encoding ignores the **order** of words (this is called a "Bag of Words" approach).*

---

## 🛠️ How to implement with Pandas
If you have a DataFrame of tokens, you can use `pd.get_dummies()` to create these vectors instantly.

# # Text to DataFrame (Long Format)
In the early stages of a pipeline, we often convert tokens into a **Pandas DataFrame**.

* **Why?**: It allows us to use powerful tools like `.value_counts()` to see which words are most popular or `.groupby()` to analyze words per document.
* **The Zip Trap**: `zip()` matches items by index. If your lists aren't the same length, Python will truncate the longer list without warning!

In [ ]:
import nltk
#nltk.download('punkt')
import pandas as pd
from nltk.tokenize import word_tokenize

# create two short documents.
corpus=['dog bites man','man bites dog'] 
ids = [11, 22, 33, 44, 55, 66, 77]
s=[]
for doc in corpus:
    #Instead of creating a list of lists (like append would do),
    # extend puts every word into one "giant bucket."
    s.extend(word_tokenize(doc))
print(s)

['dog', 'bites', 'man', 'man', 'bites', 'dog']


In [ ]:
df = pd.DataFrame(list(zip(ids, s)),columns=['Ids', 'tokens'])
df
#When you use zip(), Python stops as soon as the shortest list runs out

,Ids,tokens
0,11,dog
1,22,bites
2,33,man
3,44,man
4,55,bites
5,66,dog


In [ ]:
y = pd.get_dummies(df.tokens)
print(y)

   bites    dog    man
0  False   True  False
1   True  False  False
2  False  False   True
3  False  False   True
4   True  False  False
5  False   True  False


##  Bag of words using CountVectorizer

# 📊 Vectorization: CountVectorizer (Bag of Words)

**CountVectorizer** is a tool from the `sklearn` library that converts a collection of text documents into a matrix of token counts.

---

## ⚙️ How it Works
1. **Tokenization**: It breaks the strings into words (lowercasing them by default).
2. **Vocabulary Building**: It identifies every unique word across all documents.
3. **Encoding**: It creates a table where:
   - **Rows** = Documents
   - **Columns** = Unique words from the vocabulary
   - **Cells** = The number of times that word appears in that document.

---

## 🛠️ Key Code Methods
* `fit()`: Learns the vocabulary (the unique words).
* `transform()`: Encodes the documents into numbers based on the learned vocabulary.
* `get_feature_names_out()`: Returns the list of words used as column headers.

---

## ⚖️ Bag of Words vs. One-Hot Encoding

| Feature | One-Hot Encoding | CountVectorizer (BoW) |
| :--- | :--- | :--- |
| **Values** | Binary (0 or 1) | Integers (0, 1, 2, 3...) |
| **Meaning** | Is the word there? | How many times is the word there? |
| **Word Order** | Discarded | Discarded |

---

## ⚠️ Important Parameters
You can customize `CountVectorizer` to clean data automatically:
* `stop_words='english'`: Automatically removes common English stop words.
* `ngram_range=(1, 2)`: Captures single words AND pairs of words (e.g., "the cat", "cat sat").
* `lowercase=True`: Converts everything to lowercase (default).


Notice that words like "on", "and", and "the" have the highest numbers in your table. In NLP, we call these "High-frequency, low-info" words.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

corpus=['the cat sat on the hat','the dog ate the cat and the hat']
cv=CountVectorizer()
#scans your text to build a vocabulary of all unique words
#counts how many times each word appears in each document.
x=cv.fit_transform(corpus)

#x.toarray(): Scikit-Learn stores the result as a "Sparse Matrix"
# (to save memory). This converts it back into a dense array so Pandas can read it.

#cv.get_feature_names_out(): This retrieves the column names (the vocabulary)
# in the exact order the vectorizer placed them.

pd.DataFrame(x.toarray(),columns=cv.get_feature_names_out())

,and,ate,cat,dog,hat,on,sat,the
0,0,0,1,0,1,1,1,2
1,1,1,1,1,1,0,0,3


##  Bag of N-grams using CountVectorizer

# 🔗 N-grams: Capturing Context

Standard Bag of Words models lose all information about word order. **N-grams** fix this by grouping adjacent words together.

---

## 🔢 Definitions
* **Unigram (n=1)**: Single words. (e.g., "cat")
* **Bigram (n=2)**: Two-word sequences. (e.g., "the cat")
* **Trigram (n=3)**: Three-word sequences. (e.g., "the cat sat")

---

## 🛠️ The `ngram_range` Parameter
The `ngram_range=(min_n, max_n)` argument tells the vectorizer which lengths to include.

* `ngram_range=(1, 1)`: Only unigrams (Default).
* `ngram_range=(2, 2)`: Only bigrams.
* `ngram_range=(1, 3)`: Everything from single words up to 3-word phrases.

---

## ⚖️ Trade-offs

| Feature | Unigrams Only | N-grams (1, 3) |
| :--- | :--- | :--- |
| **Context** | Low (ignores order) | High (captures phrases) |
| **Feature Count** | Small (number of unique words) | Massive (exponentially more columns) |
| **Sparsity** | High | Extremely High (most cells will be 0) |

---

## 💡 Example Result
In your specific corpus, the phrase **"the cat ate the cat"** will generate unique features like:
- `the cat ate`
- `cat ate the`
- `ate the cat`

In [ ]:

from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
corpus=['the cat sat on the hat','the cat ate the cat and the hat']

# By adding ngram_range=(1, 3), you told the vectorizer to look for:
#   Unigrams: Single words (e.g., "cat", "hat").
#   Bigrams: Pairs of consecutive words (e.g., "the cat", "cat sat").
#   Trigrams: Groups of three consecutive words (e.g., "the cat sat").

cv=CountVectorizer(ngram_range=(1, 3))
x=cv.fit_transform(corpus)
pd.DataFrame(x.toarray(),columns=cv.get_feature_names_out())

,and,and the,and the hat,ate,ate the,ate the cat,cat,cat and,cat and the,cat ate,...,on the hat,sat,sat on,sat on the,the,the cat,the cat and,the cat ate,the cat sat,the hat
0,0,0,0,0,0,0,1,0,0,0,...,1,1,1,1,2,1,0,0,1,1
1,1,1,1,1,1,1,2,1,1,1,...,0,0,0,0,3,2,1,1,0,1


##  TFIDF

# ⚖️ TF-IDF: Importance-Based Vectorization

**TF-IDF** stands for **Term Frequency-Inverse Document Frequency**. It is a statistical measure used to evaluate how relevant a word is to a document in a collection. TF-IDF penalizes "boring" words like "the" and "and" that appear everywhere, and boosts "interesting" words like "ate" or "sat" that appear only in specific documents.

---

## 🧪 The Formula Logic
TF-IDF is calculated by multiplying two metrics:

1.  **Term Frequency (TF)**: How often a word appears in a document.
    * *Logic*: If "cat" appears 5 times in Doc A, it's likely important to Doc A.
2.  **Inverse Document Frequency (IDF)**: How unique a word is across all documents.
    * *Logic*: If "the" appears in **every** document, its importance score is lowered. If "ate" appears in only **one** document, its importance score is raised.

---

## 📊 Interpreting the Values
* **Values near 0**: Words that appear in almost all documents (e.g., stop words) or don't appear in that document at all.
* **Higher values**: Words that are frequent in a specific document but rare in the rest of the corpus (the "keywords").

---

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
corpus=['the cat sat on the hat','the dog ate the cat and the hat']
cv=TfidfVectorizer()
x=cv.fit_transform(corpus)
pd.DataFrame(x.toarray(),columns=cv.get_feature_names_out())

,and,ate,cat,dog,hat,on,sat,the
0,0.00000,0.00000,0.317011,0.00000,0.317011,0.445548,0.445548,0.634021
1,0.34162,0.34162,0.243065,0.34162,0.243065,0.000000,0.000000,0.729196


1.  **Manual Mapping** (Pandas zip/dummies)
2.  **One-Hot Encoding** (Binary)
3.  **Bag of Words** (CountVectorizer)
4.  **TF-IDF** (Weighted importance)

### we can use N-grams using TFIDF too

## Text Similarity

# 📐 Measuring Similarity: Cosine Similarity

Once text is converted into vectors (using CountVectorizer or TF-IDF), we need a way to compare them. **Cosine Similarity** is the standard metric for this.

---

## 🧐 Why not use Euclidean Distance?
Euclidean distance measures the straight-line distance between points. In NLP, this fails because:
* If **Doc A** is "I love cats" (short) 
* and **Doc B** is "I love cats..." repeated 100 times (long)
* Euclidean distance will say they are very different because the points are far apart. 
* **Cosine Similarity** will see the angle is 0° and correctly say they are identical in meaning.

---

## 🔢 The Values
| Score | Meaning |
| :--- | :--- |
| **1.0** | Perfect Similarity (Identical word usage) |
| **0.5** | Partial Similarity |
| **0.0** | No Similarity (Zero words in common) |

---

##  Implementation
Scikit-learn provides a optimized function for this. Remember that it expects a 2D input (Matrix).



In [ ]:
from numpy import dot
from numpy.linalg import norm

#This is the literal mathematical formula: 
# The Dot Product divided by the product of the magnitudes.
cosine=lambda v1, v2: dot(v1,v2)/(norm(v1) * norm(v2))
cosine([1,1,1,0],[1,1,0,1])

np.float64(0.6666666666666667)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

#Note that it expects 2D arrays (hence the double brackets [[ ]])
cosine_similarity([[1,1,1,0]],[[1,1,0,1]])

array([[0.66666667]])

In [ ]:
x.toarray()

array([[0.        , 0.        , 0.31701073, 0.        , 0.31701073,
        0.44554752, 0.44554752, 0.63402146],
       [0.34161973, 0.34161973, 0.24306525, 0.34161973, 0.24306525,
        0.        , 0.        , 0.72919575]])

### We can access the feature vectors from x directly and measure their similarity

In [ ]:
#This compares the first document in your TF-IDF matrix
#  (x[0]) to the second (x[1]).
cosine_similarity(x[0],x[1])

array([[0.61643434]])

## Words Embedding Vectors Using word2vec

# 🌐 Word Embeddings: Word2Vec

**Word2Vec** is a technique to represent words as dense vectors in a continuous vector space. It captures **semantic meaning**—the computer actually "understands" that "king" is to "man" what "queen" is to "woman."

---

## 🏗️ The CBOW Model (Continuous Bag of Words)
Your code uses the default Word2Vec architecture called **CBOW**.
* **Goal**: Predict the current word using the surrounding window of words.
* **Example**: "The cat ___ on the mat." -> The model learns to predict "sat."

---

##  Key Hyperparameters in your Code
* `min_count = 1`: Tells the model to include every word that appears at least once. (In large projects, we usually set this to 5+ to ignore rare typos).
* `vector_size = 100`: Each word is turned into a list of 100 numbers. More dimensions can capture more complex meanings but require more data.
* `window = 5`: The model looks at 5 words to the left and 5 words to the right of the target word to learn context.

---

## 📉 Interpreting your Results
* **Similarity('alice', 'wonderland')**: Should be **high** (closer to 1.0) because they frequently appear together in the same sentences in this specific book.
* **Similarity('alice', 'machines')**: Should be **low** because "machines" is not a common context word in *Alice in Wonderland*.

---
# 🤖 Understanding the Word2Vec Model Call

When you call `gensim.models.Word2Vec()`, you are defining how the neural network "sees" the language.

## The Architecture: CBOW vs Skip-gram
By default (as in your code), Gensim uses **CBOW (Continuous Bag of Words)**.
### 1. CBOW (Continuous Bag of Words)
* **Logic**: Predicts the **Middle** word from the **Context**.
* **Best for**: Small datasets or when training speed is a priority.
* **Result**: Works very well with frequent words (e.g., "the", "is", "cat") but can struggle with rare words because it "averages" the context.

### 2. Skip-gram
* **Logic**: Predicts the **Context** from the **Middle** word.
* **Best for**: Large datasets where you want high accuracy for infrequent/rare words.
* **Result**: Since it predicts context from a single word, it gets many more training "samples" from the same sentence, leading to better semantic embeddings for rare terms.

### The Training Process
1. **Vocabulary Building**: The model scans `data` to find all unique words appearing at least `min_count` times.
2. **Weight Training**: The model moves a "sliding window" across the text. It adjusts the 100 numbers in each word's vector until words that appear in similar contexts have similar numbers.
3. **The Result**: You get a **Word Embedding** matrix where the "distance" between vectors represents semantic similarity.

### Key Methods used after Training:
* `model.wv.similarity(w1, w2)`: Calculates the **Cosine Similarity** between the trained vectors of two words.
* `model.wv.most_similar(w1)`: Finds the words that are mathematically "closest" to the input word.

## 🚀 Why Embeddings are better than TF-IDF
1. **Dense Vectors**: Instead of 10,000 columns (mostly zeros), every word is a compact list of 100-300 numbers.
2. **Semantic Similarity**: TF-IDF sees "happy" and "joyful" as totally different. Word2Vec realizes they are similar because they are surrounded by similar words.

In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize
import warnings
warnings.filterwarnings(action = 'ignore')
import gensim
from gensim.models import Word2Vec
#  Reads ‘alice.txt’ file
sample = open("alice.txt", "r", encoding="latin-1")
s = sample.read()

# Replaces escape character with space
f = s.replace("\n", " ")

data = []

# iterate through each sentence in the file
for i in sent_tokenize(f):
    temp = []

    # tokenize the sentence into words
    for j in word_tokenize(i):
        temp.append(j.lower())

    data.append(temp)

# Create CBOW model
model1 = gensim.models.Word2Vec(data, min_count = 1,
                              vector_size = 100, window = 5)

# Print results
print("Cosine similarity between 'alice' " + "and 'wonderland' - CBOW : ", model1.wv.similarity('alice', 'wonderland'))

print("Cosine similarity between 'alice' " + "and 'machines' - CBOW : ", model1.wv.similarity('alice', 'machines'))

Cosine similarity between 'alice' and 'wonderland' - CBOW :  0.98552835
Cosine similarity between 'alice' and 'machines' - CBOW :  0.8266414


In [14]:
# Create Skip Gram model
model2 = gensim.models.Word2Vec(data, min_count = 1,  vector_size = 100, window = 5, sg = 1)

# Print results
print("Cosine similarity between 'alice' " + "and 'wonderland' - Skip Gram : ", model2.wv.similarity('alice', 'wonderland'))

print("Cosine similarity between 'alice' " + "and 'machines' - Skip Gram : ", model2.wv.similarity('alice', 'machines'))

Cosine similarity between 'alice' and 'wonderland' - Skip Gram :  0.60836804
Cosine similarity between 'alice' and 'machines' - Skip Gram :  0.7731375
